# 03 - Modeling on Imbalanced Data

This notebook trains multiple models and compares them using PR-AUC, precision, recall, and tuned thresholds.

In [ ]:
from pathlib import Path
import pandas as pd

from src.preprocessing import SplitConfig, split_train_test
from src.model_training import ModelTrainer, TrainingConfig

df = pd.read_csv(Path("../data/raw/creditcard.csv"))

# Optional: speed-up for experimentation in notebook
sample_size = 120000
if sample_size < len(df):
    df = (
        df.groupby('Class', group_keys=False)
          .apply(lambda g: g.sample(frac=sample_size/len(df), random_state=42))
          .reset_index(drop=True)
    )

X_train, X_test, y_train, y_test = split_train_test(df, split_config=SplitConfig(test_size=0.2, random_state=42))
trainer = ModelTrainer(TrainingConfig(random_state=42, min_recall_for_threshold=0.60, enable_smoteenn=True))
outputs = trainer.train_and_compare(X_train, y_train, X_test, y_test)
comparison_df = outputs['comparison_df']
comparison_df

In [ ]:
best_model_name = outputs['best_model_name']
best_threshold = outputs['best_threshold']
print('Best model:', best_model_name)
print('Tuned threshold:', round(best_threshold, 4))

In [ ]:
from pathlib import Path
from IPython.display import Image, display

display(Image(filename=str(Path('../artifacts/plots/model_comparison_pr_auc.png'))))
display(Image(filename=str(Path('../artifacts/plots/best_model_confusion_matrix.png'))))
display(Image(filename=str(Path('../artifacts/plots/best_model_pr_curve.png'))))

## Why PR-AUC matters

For rare-event fraud detection, PR-AUC focuses on minority-class retrieval quality. ROC-AUC can look strong even when fraud precision is weak at deployable thresholds.